# 授業方略実験ノートブック

最終更新: 2026-07-13 14:15 JST

このノートブックでは、教師AIが授業手法を考える前段階を確認します。生徒AIの発話を作り、伝達AIが観察し、教師AI用のコンテキストを作成して、次の授業方略を選びます。

## 1. GitHubから最新版を取得

Colabでは最初にこのセルを実行します。まだcloneしていない場合はcloneし、すでにclone済みの場合は最新版をpullします。

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/Hiromu-0219/student-ai-test.git"
PROJECT_DIR = Path("/content/student-ai")

if not PROJECT_DIR.exists():
    !git clone {REPO_URL} {PROJECT_DIR}

%cd /content/student-ai
!git status --short --branch
!git pull
!git log -1 --oneline

## 2. 依存関係をインストール

In [ ]:
!pip install -q -r requirements.txt

## 3. 実験設定

軽く確認する場合は `USE_MOCK_MODEL = True` のまま使います。実LLMで生徒発話を生成したい場合は `False` に変更します。

In [ ]:
STUDENT_ID = "S002"
USE_MOCK_MODEL = True
TEACHER_MESSAGE = "x + 3 = 8 を解いてみましょう。+3を反対側へ移すと、符号はどうなりますか。"

print("student_id:", STUDENT_ID)
print("use_mock_model:", USE_MOCK_MODEL)
print("teacher_message:", TEACHER_MESSAGE)

## 4. 生徒AIの発話を生成

ここでは授業中の対話として生徒発話を生成します。実験を繰り返しやすくするため、標準では知識状態を更新しません。

In [ ]:
from src.student_ai import StudentAISimulator

sim = StudentAISimulator(use_mock_model=USE_MOCK_MODEL)
student_record = sim.respond(
    STUDENT_ID,
    TEACHER_MESSAGE,
    update_knowledge=False,
)
student_utterance = student_record["answer"]

print(student_utterance)

## 5. 伝達AIが発話を観察

伝達AIが、生徒発話から見える個人特徴を推定し、教師AIに渡す注意点を作ります。

In [ ]:
from pprint import pprint

from src.observer import CommunicationAI

communication_ai = CommunicationAI()
observation = communication_ai.classify_utterance(
    utterance=student_utterance,
    student_id=STUDENT_ID,
).to_dict()

pprint(observation)

## 6. 教師AI用コンテキストを作成

教師AI用コンテキストには、生徒状態、単元目標、直近の生徒発話、伝達AIの観察結果、次に出せる問題が入ります。

In [ ]:
from pprint import pprint

from src.state_manager import StateManager
from src.teacher import TeacherContextBuilder

state_manager = StateManager()
student_state = state_manager.load_student(STUDENT_ID)

context_builder = TeacherContextBuilder()
teacher_context = context_builder.build_context(
    student_state=student_state,
    recent_student_utterance=student_utterance,
    communication_observation=observation,
)

pprint({
    "target_skill": teacher_context["target_skill"],
    "lesson_goal": teacher_context["lesson_goal"],
    "student_state_summary": teacher_context["student_state_summary"],
    "communication_ai_observation": teacher_context["communication_ai_observation"],
})

## 7. 授業方略を選択

現在のMVPでは、判断理由を確認しやすいようにルールベースで授業方略を選びます。将来的には同じコンテキストをLLM教師に渡せます。

In [ ]:
from pprint import pprint

from src.teacher import RuleBasedTeachingStrategySelector

selector = RuleBasedTeachingStrategySelector()
teacher_decision = selector.select_strategy(teacher_context)

pprint(teacher_decision)

## 8. 最新結果を保存

あとで比較できるように、教師AI用コンテキストと授業方略の判断結果をJSONで保存します。

In [ ]:
import json
from pathlib import Path

output_dir = Path("data/assessments")
output_dir.mkdir(parents=True, exist_ok=True)

context_path = output_dir / "teacher_context_latest.json"
decision_path = output_dir / "teaching_strategy_decision_latest.json"

context_path.write_text(json.dumps(teacher_context, ensure_ascii=False, indent=2), encoding="utf-8")
decision_path.write_text(json.dumps(teacher_decision, ensure_ascii=False, indent=2), encoding="utf-8")

print("saved:", context_path)
print("saved:", decision_path)